# Analyse des Concentrations de Planctons Marins
## Jalon 3/3 : Nouveaux Algorithmes de Clustering


---

### Résumé
Ce notebook constitue le troisième et dernier jalon du projet. Il applique des algorithmes
de clustering avancés (DBSCAN, HDBSCAN, OPTICS, Spectral Clustering, GMM) sur les données
planctoniques et compare leurs performances avec la baseline K-means du Jalon 1.


## Table des matières

1. [Chargement des résultats Jalon 1](#chargement)
2. [DBSCAN](#dbscan)
3. [HDBSCAN](#hdbscan)
4. [OPTICS](#optics)
5. [Spectral Clustering](#spectral)
6. [Gaussian Mixture Models (GMM)](#gmm)
7. [Comparaison globale des clusterings](#comparaison)
8. [Conclusions générales](#conclusions)


<a id="chargement"></a>

---

## 1. Chargement des résultats Jalon 1


In [ ]:
# === Chargement des résultats Jalon 1 ===
import pickle

with open('data/jalon1_outputs.pkl', 'rb') as f:
    _d = pickle.load(f)

df            = _d['df']
df_hellinger  = _d['df_hellinger']
COLS_CONC     = _d['COLS_CONC']
COLS_ENV      = _d['COLS_ENV']
X_pca         = _d['X_pca']
n_80          = _d['n_80']
pca           = _d['pca']
labels_final  = _d['labels_final']
best_sil_k    = _d['best_sil_k']
K_OPTIMAL     = _d['K_OPTIMAL']
unique_layers = _d['unique_layers']
X_hell        = df_hellinger.values
X_clust       = X_pca[:, :n_80]  # Espace de clustering (même que Jalon 1)

print(f"Données Jalon 1 chargées — {X_pca.shape[0]} obs, {n_80} composantes PCA")
print(f"K-means baseline : {K_OPTIMAL} clusters, silhouette = {best_sil_k:.4f}")


In [ ]:
# === Imports Jalon 3 ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings("ignore")

from sklearn.cluster import DBSCAN, OPTICS, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score as ari_score, normalized_mutual_info_score as nmi_score, v_measure_score
from sklearn.preprocessing import LabelEncoder

# Constants from Jalon 1
RANDOM_STATE = 42
cluster_colors = _d.get('cluster_colors', plt.cm.Set1(np.linspace(0, 0.9, K_OPTIMAL)))

# Global storage for results
metrics_all = {}
all_clusterings = {}

def compute_metrics(X, labels, labels_ref=None):
    """Calcule les métriques internes et externes."""
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    mask = labels != -1
    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = silhouette_score(X[mask], labels[mask], sample_size=min(1000, mask.sum()), random_state=42)
        db  = davies_bouldin_score(X[mask], labels[mask])
        ch  = calinski_harabasz_score(X[mask], labels[mask])
    else:
        sil, db, ch = np.nan, np.nan, np.nan
    res = {'n_clusters': n_clusters, 'n_noise': n_noise, 'silhouette': sil, 'davies_bouldin': db, 'calinski_harabasz': ch}
    if labels_ref is not None:
        res['ari'] = ari_score(labels_ref, labels)
        res['nmi'] = nmi_score(labels_ref, labels)
        res['v_measure'] = v_measure_score(labels_ref, labels)
    return res

# Initialisation avec K-means (Jalon 1)
metrics_all['K-means'] = compute_metrics(X_clust, labels_final, labels_final)
all_clusterings['K-means'] = labels_final

# Logic HDBSCAN
try:
    import hdbscan as hdbscan_lib
    HDBSCAN_LIB = True
except ImportError:
    try:
        from sklearn.cluster import HDBSCAN as hdbscan_lib
        HDBSCAN_LIB = False
    except ImportError:
        hdbscan_lib = None
        HDBSCAN_LIB = False


## 4.1 DBSCAN : Density-Based Spatial Clustering of Applications with Noise (Ester et al., 1996)

DBSCAN classe les points selon leur densité locale. Il identifie trois types de points :
- **Points cœurs** : au moins `min_samples` voisins dans un rayon ε
- **Points frontières** : dans le voisinage d'un cœur, mais pas cœurs eux-mêmes
- **Points de bruit** : isolés, sans appartenance à un cluster (label = -1)

**Avantages :** pas de k à fixer, détecte le bruit, formes arbitraires  
**Limites :** sensible à ε, difficultés avec des densités variables

Le paramètre ε est estimé via le **graphe des distances au k-ème voisin** : le "coude" indique le seuil de séparation densité/bruit.

In [ ]:
# === DBSCAN ===

N_DBSCAN = min(5000, len(X_clust))
rng_db   = np.random.default_rng(42)
idx_db   = rng_db.choice(len(X_clust), N_DBSCAN, replace=False)
X_db     = X_clust[idx_db]

# --- 1. Graphe k-NN pour estimer epsilon ---
k_nn              = 5
nbrs              = NearestNeighbors(n_neighbors=k_nn).fit(X_db)
knn_distances, _  = nbrs.kneighbors(X_db)
knn_dists_sorted  = np.sort(knn_distances[:, -1])

# Candidats epsilon = percentiles de la distribution des distances k-NN
eps_candidates    = np.percentile(knn_dists_sorted, [70, 80, 90, 95])
min_samples_vals  = [3, 5, 10, 15]

# --- 2. Grid search ---
db_results = []
for eps in eps_candidates:
    for ms in min_samples_vals:
        db   = DBSCAN(eps=eps, min_samples=ms).fit(X_db)
        labs = db.labels_
        n_cl = len(set(labs)) - (1 if -1 in labs else 0)
        noise_p = (labs == -1).sum() / len(labs) * 100
        if n_cl >= 2 and (labs != -1).sum() > n_cl:
            sil = silhouette_score(X_db[labs != -1], labs[labs != -1],
                                   sample_size=min(1000, (labs != -1).sum()), random_state=42)
        else:
            sil = np.nan
        db_results.append({'eps': round(eps, 3), 'min_samples': ms,
                           'n_clusters': n_cl, 'noise_pct': round(noise_p, 1),
                           'silhouette': round(sil, 4) if not np.isnan(sil) else np.nan})

df_dbscan_grid = pd.DataFrame(db_results)
print("Grid search DBSCAN :")
print(df_dbscan_grid.to_string(index=False))

# Meilleurs paramètres
valid_db = df_dbscan_grid[(df_dbscan_grid['n_clusters'] >= 2) &
                           (df_dbscan_grid['noise_pct'] < 40) &
                           (df_dbscan_grid['silhouette'].notna())]
if len(valid_db):
    best_db  = valid_db.loc[valid_db['silhouette'].idxmax()]
    BEST_EPS = best_db['eps']
    BEST_MS  = int(best_db['min_samples'])
else:
    BEST_EPS, BEST_MS = eps_candidates[1], 5

print(f"\nMeilleurs paramètres : eps={BEST_EPS}, min_samples={BEST_MS}")

# Application sur données complètes
dbscan_full     = DBSCAN(eps=BEST_EPS, min_samples=BEST_MS).fit(X_clust)
labels_dbscan   = dbscan_full.labels_
n_cl_db         = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
noise_db        = (labels_dbscan == -1).sum()
print(f"  → {n_cl_db} clusters, {noise_db} bruit ({noise_db/len(labels_dbscan)*100:.1f}%)")

metrics_all['DBSCAN']      = compute_metrics(X_clust, labels_dbscan, labels_final)
all_clusterings['DBSCAN']  = labels_dbscan

# --- Visualisation ---
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# a. k-NN distance plot
axes[0].plot(knn_dists_sorted, color='steelblue', lw=1.5)
axes[0].axhline(BEST_EPS, color='crimson', linestyle='--', label=f'ε={BEST_EPS:.3f}')
axes[0].set_xlabel("Points triés", fontsize=12)
axes[0].set_ylabel(f"Distance au {k_nn}e voisin", fontsize=12)
axes[0].set_title("Graphe k-NN (choix de ε)", fontsize=13)
axes[0].legend(fontsize=11); axes[0].grid(alpha=0.3)

# b. Clusters PC1-PC2
for lbl in sorted(set(labels_dbscan)):
    mask = labels_dbscan == lbl
    if lbl == -1:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c='lightgray', s=5, alpha=0.3, label='Bruit')
    else:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[plt.cm.tab10(lbl % 10)],
                       s=15, alpha=0.6, label=f'C{lbl+1}')
axes[1].set_xlabel("PC1", fontsize=12); axes[1].set_ylabel("PC2", fontsize=12)
axes[1].set_title(f"DBSCAN — {n_cl_db} clusters (ε={BEST_EPS:.3f}, ms={BEST_MS})", fontsize=13)
if n_cl_db <= 10:
    axes[1].legend(fontsize=9, markerscale=2)
axes[1].grid(alpha=0.2)

# c. Profil biologique par cluster
if n_cl_db >= 2:
    df_db_bio = df_hellinger.copy()
    df_db_bio['cluster'] = labels_dbscan
    df_db_bio = df_db_bio[df_db_bio['cluster'] != -1]
    bio_db = df_db_bio.groupby('cluster')[COLS_CONC].mean()
    im = axes[2].imshow(bio_db.values.T, aspect='auto', cmap='YlOrRd')
    axes[2].set_xticks(range(len(bio_db)))
    axes[2].set_xticklabels([f'C{i+1}' for i in bio_db.index], fontsize=10)
    axes[2].set_yticks(range(len(COLS_CONC)))
    axes[2].set_yticklabels(COLS_CONC, fontsize=8)
    plt.colorbar(im, ax=axes[2], label='Hellinger (moy.)')
    axes[2].set_title("Profil biologique DBSCAN", fontsize=13)

plt.tight_layout()
plt.savefig('figures/fig_19_dbscan.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nMétriques DBSCAN :")
print(f"  Silhouette     : {metrics_all['DBSCAN']['silhouette']:.4f}")
print(f"  Davies-Bouldin : {metrics_all['DBSCAN']['davies_bouldin']:.4f}")
print(f"  Calinski-Harab.: {metrics_all['DBSCAN']['calinski_harabasz']:.1f}")
print(f"  ARI vs K-means : {metrics_all['DBSCAN']['ari']:.4f}")
print(f"  Points de bruit: {noise_db} ({noise_db/len(labels_dbscan)*100:.1f}%)")

## 4.2 HDBSCAN : Hierarchical DBSCAN (Campello et al., 2013)

HDBSCAN étend DBSCAN en construisant une **hiérarchie de clusters** basée sur la densité, puis en extrayant les clusters les plus stables selon un critère de persistance. Il est plus robuste que DBSCAN face aux variations de densité et ne nécessite pas de fixer ε.

**Paramètres clés :**
- `min_cluster_size` : taille minimale d'un cluster stable
- `min_samples` : robustesse au bruit (optionnel, par défaut = `min_cluster_size`)

**Avantages :** gère les densités variables, hiérarchie extraite automatiquement, robuste  
**Limites :** plus lent sur grands jeux de données, sensible à `min_cluster_size`

In [ ]:
# === HDBSCAN ===

mcs_vals     = [10, 20, 50, 100, 200]
ms_vals_hdb  = [None, 5, 10]
hdb_results  = []

N_HDB    = min(5000, len(X_clust))
rng_hdb  = np.random.default_rng(42)
idx_hdb  = rng_hdb.choice(len(X_clust), N_HDB, replace=False)
X_hdb    = X_clust[idx_hdb]

if hdbscan_lib is not None:
    print("Grid search HDBSCAN (min_cluster_size × min_samples)...")
    for mcs in mcs_vals:
        for ms in ms_vals_hdb:
            try:
                if HDBSCAN_LIB:
                    params = {'min_cluster_size': mcs}
                    if ms is not None:
                        params['min_samples'] = ms
                    clusterer = hdbscan_lib.HDBSCAN(**params)
                    clusterer.fit(X_hdb)
                    labs = clusterer.labels_
                else:
                    params = {'min_cluster_size': mcs}
                    if ms is not None:
                        params['min_samples'] = ms
                    clusterer = hdbscan_lib(**params)
                    labs = clusterer.fit_predict(X_hdb)

                n_cl     = len(set(labs)) - (1 if -1 in labs else 0)
                noise_p  = (labs == -1).sum() / len(labs) * 100
                if n_cl >= 2 and (labs != -1).sum() > n_cl:
                    sil = silhouette_score(X_hdb[labs != -1], labs[labs != -1],
                                          sample_size=min(1000, (labs != -1).sum()), random_state=42)
                else:
                    sil = np.nan
                hdb_results.append({'min_cluster_size': mcs, 'min_samples': ms or 'auto',
                                    'n_clusters': n_cl, 'noise_pct': round(noise_p, 1),
                                    'silhouette': round(sil, 4) if not np.isnan(sil) else np.nan})
            except Exception:
                hdb_results.append({'min_cluster_size': mcs, 'min_samples': ms or 'auto',
                                    'n_clusters': -1, 'noise_pct': np.nan, 'silhouette': np.nan})

    df_hdb_grid = pd.DataFrame(hdb_results)
    print(df_hdb_grid.to_string(index=False))

    valid_hdb = df_hdb_grid[(df_hdb_grid['n_clusters'] >= 2) &
                             (df_hdb_grid['noise_pct'] < 50) &
                             (df_hdb_grid['silhouette'].notna())]
    if len(valid_hdb):
        best_h      = valid_hdb.loc[valid_hdb['silhouette'].idxmax()]
        BEST_MCS    = int(best_h['min_cluster_size'])
        BEST_MS_HDB = None if best_h['min_samples'] == 'auto' else int(best_h['min_samples'])
    else:
        BEST_MCS, BEST_MS_HDB = 50, None

    print(f"\nMeilleurs paramètres : min_cluster_size={BEST_MCS}, min_samples={BEST_MS_HDB}")

    # Application sur données complètes
    try:
        if HDBSCAN_LIB:
            pf = {'min_cluster_size': BEST_MCS}
            if BEST_MS_HDB:
                pf['min_samples'] = BEST_MS_HDB
            hdb_clf = hdbscan_lib.HDBSCAN(**pf)
            hdb_clf.fit(X_clust)
            labels_hdbscan = hdb_clf.labels_
        else:
            pf = {'min_cluster_size': BEST_MCS}
            if BEST_MS_HDB:
                pf['min_samples'] = BEST_MS_HDB
            labels_hdbscan = hdbscan_lib(**pf).fit_predict(X_clust)
    except Exception as e:
        print(f"Erreur sur données complètes : {e}")
        labels_hdbscan = np.full(len(X_clust), -1)

    n_cl_hdb  = len(set(labels_hdbscan)) - (1 if -1 in labels_hdbscan else 0)
    noise_hdb = (labels_hdbscan == -1).sum()
    print(f"  → {n_cl_hdb} clusters, {noise_hdb} bruit ({noise_hdb/len(labels_hdbscan)*100:.1f}%)")

    metrics_all['HDBSCAN']      = compute_metrics(X_clust, labels_hdbscan, labels_final)
    all_clusterings['HDBSCAN']  = labels_hdbscan

    # --- Visualisation ---
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    # a. Grid search heatmap
    mcs_list = sorted(df_hdb_grid['min_cluster_size'].unique())
    ms_list  = df_hdb_grid['min_samples'].unique().tolist()
    pivot_n  = df_hdb_grid.pivot(index='min_samples', columns='min_cluster_size', values='n_clusters').fillna(0)
    im = axes[0].imshow(pivot_n.values, aspect='auto', cmap='YlOrRd')
    axes[0].set_xticks(range(len(mcs_list)))
    axes[0].set_xticklabels(mcs_list, fontsize=9)
    axes[0].set_yticks(range(len(pivot_n.index)))
    axes[0].set_yticklabels(pivot_n.index, fontsize=9)
    axes[0].set_xlabel("min_cluster_size", fontsize=11)
    axes[0].set_ylabel("min_samples", fontsize=11)
    for (i, j), val in np.ndenumerate(pivot_n.values):
        axes[0].text(j, i, f'{int(val)}', ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=axes[0], label='n_clusters')
    axes[0].set_title("HDBSCAN — Grid search n_clusters", fontsize=12)

    # b. Clusters PC1-PC2
    for lbl in sorted(set(labels_hdbscan)):
        mask = labels_hdbscan == lbl
        if lbl == -1:
            axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c='lightgray', s=5, alpha=0.3, label='Bruit')
        else:
            axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[plt.cm.tab10(lbl % 10)],
                           s=15, alpha=0.6, label=f'C{lbl+1}')
    axes[1].set_xlabel("PC1", fontsize=12); axes[1].set_ylabel("PC2", fontsize=12)
    axes[1].set_title(f"HDBSCAN — {n_cl_hdb} clusters (mcs={BEST_MCS})", fontsize=13)
    if n_cl_hdb <= 10:
        axes[1].legend(fontsize=9, markerscale=2)
    axes[1].grid(alpha=0.2)

    # c. Profil biologique
    if n_cl_hdb >= 2:
        df_hdb_bio = df_hellinger.copy()
        df_hdb_bio['cluster'] = labels_hdbscan
        df_hdb_bio = df_hdb_bio[df_hdb_bio['cluster'] != -1]
        bio_hdb = df_hdb_bio.groupby('cluster')[COLS_CONC].mean()
        im2 = axes[2].imshow(bio_hdb.values.T, aspect='auto', cmap='YlOrRd')
        axes[2].set_xticks(range(len(bio_hdb)))
        axes[2].set_xticklabels([f'C{i+1}' for i in bio_hdb.index], fontsize=10)
        axes[2].set_yticks(range(len(COLS_CONC)))
        axes[2].set_yticklabels(COLS_CONC, fontsize=8)
        plt.colorbar(im2, ax=axes[2], label='Hellinger (moy.)')
        axes[2].set_title("Profil biologique HDBSCAN", fontsize=13)

    plt.tight_layout()
    plt.savefig('figures/fig_21_hdbscan.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f"\nMétriques HDBSCAN :")
    print(f"  Silhouette     : {metrics_all['HDBSCAN']['silhouette']:.4f}")
    print(f"  Davies-Bouldin : {metrics_all['HDBSCAN']['davies_bouldin']:.4f}")
    print(f"  Calinski-Harab.: {metrics_all['HDBSCAN']['calinski_harabasz']:.1f}")
    print(f"  ARI vs K-means : {metrics_all['HDBSCAN']['ari']:.4f}")
else:
    print("HDBSCAN non disponible.")
    labels_hdbscan = np.full(len(X_clust), -1)
    metrics_all['HDBSCAN']     = compute_metrics(X_clust, labels_hdbscan, labels_final)
    all_clusterings['HDBSCAN'] = labels_hdbscan

## 4.3 OPTICS : Ordering Points to Identify the Clustering Structure (Ankerst et al., 1999)

OPTICS généralise DBSCAN en ordonnant les points selon leur *reachability distance*, ce qui permet de détecter des **clusters de densités variables**. Le **reachability plot** est la signature visuelle de la structure hiérarchique des données.

**Paramètres clés :**
- `min_samples` : nombre minimum de points dans le voisinage (analogue à DBSCAN)
- `max_eps` : rayon maximum de recherche (∞ par défaut)

**Avantages :** multi-densité, pas d'ε à fixer, reachability plot très informatif  
**Limites :** O(n log n), extraction des clusters non triviale, sous-échantillonnage requis ici

In [ ]:
# === OPTICS ===

N_OPT    = min(4000, len(X_clust))
rng_opt  = np.random.default_rng(42)
idx_opt  = rng_opt.choice(len(X_clust), N_OPT, replace=False)
X_opt    = X_clust[idx_opt]

# Grid search min_samples
ms_optics   = [5, 10, 20, 30]
opt_results = []
print("Grid search OPTICS (min_samples)...")
for ms in ms_optics:
    opt     = OPTICS(min_samples=ms, metric='euclidean', n_jobs=-1)
    opt.fit(X_opt)
    labs_xi = opt.labels_
    n_cl_xi = len(set(labs_xi)) - (1 if -1 in labs_xi else 0)
    noise_xi = (labs_xi == -1).sum() / len(labs_xi) * 100
    if n_cl_xi >= 2 and (labs_xi != -1).sum() > n_cl_xi:
        sil_xi = silhouette_score(X_opt[labs_xi != -1], labs_xi[labs_xi != -1],
                                  sample_size=min(1000, (labs_xi != -1).sum()), random_state=42)
    else:
        sil_xi = np.nan
    opt_results.append({'min_samples': ms, 'n_clusters': n_cl_xi,
                        'noise_pct': round(noise_xi, 1),
                        'silhouette': round(sil_xi, 4) if not np.isnan(sil_xi) else np.nan})
    print(f"  ms={ms:2d} → {n_cl_xi} clusters, {noise_xi:.1f}% bruit")

df_optics_grid = pd.DataFrame(opt_results)
valid_opt  = df_optics_grid[(df_optics_grid['n_clusters'] >= 2) &
                             (df_optics_grid['noise_pct'] < 50) &
                             (df_optics_grid['silhouette'].notna())]
BEST_MS_OPT = int(valid_opt.loc[valid_opt['silhouette'].idxmax(), 'min_samples']) if len(valid_opt) else 10
print(f"\nMeilleur min_samples OPTICS : {BEST_MS_OPT}")

# Application avec meilleur paramètre (même sous-échantillon)
optics_final    = OPTICS(min_samples=BEST_MS_OPT, metric='euclidean', n_jobs=-1)
optics_final.fit(X_opt)
labels_optics_sub = optics_final.labels_

# Extension aux données complètes
if N_OPT < len(X_clust):
    from sklearn.neighbors import KNeighborsClassifier as _KNN
    _knn_opt = _KNN(n_neighbors=5)
    _knn_opt.fit(X_opt, labels_optics_sub)
    labels_optics = _knn_opt.predict(X_clust)
else:
    labels_optics = labels_optics_sub

n_cl_opt  = len(set(labels_optics)) - (1 if -1 in labels_optics else 0)
noise_opt = (labels_optics == -1).sum()
print(f"  → {n_cl_opt} clusters, {noise_opt} bruit ({noise_opt/len(labels_optics)*100:.1f}%)")

metrics_all['OPTICS']      = compute_metrics(X_clust, labels_optics, labels_final)
all_clusterings['OPTICS']  = labels_optics

# --- Visualisation ---
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# a. Reachability plot
order           = optics_final.ordering_
reach           = optics_final.reachability_[order]
labs_ordered    = labels_optics_sub[order]
unique_opt_labs = sorted(set(labels_optics_sub[labels_optics_sub != -1]))

for lbl in unique_opt_labs:
    mask_r = labs_ordered == lbl
    axes[0].vlines(np.where(mask_r)[0], 0, reach[mask_r],
                   colors=[plt.cm.tab10(lbl % 10)], alpha=0.7, label=f'C{lbl+1}')
mask_noise_r = labs_ordered == -1
axes[0].vlines(np.where(mask_noise_r)[0], 0, reach[mask_noise_r], colors='lightgray', alpha=0.3)
axes[0].set_xlabel("Points (ordonnés)", fontsize=11)
axes[0].set_ylabel("Reachability distance", fontsize=11)
axes[0].set_title(f"OPTICS — Reachability plot (ms={BEST_MS_OPT})", fontsize=12)
if len(unique_opt_labs) <= 8:
    axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.2)

# b. Clusters PC1-PC2
for lbl in sorted(set(labels_optics)):
    mask = labels_optics == lbl
    if lbl == -1:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c='lightgray', s=5, alpha=0.3, label='Bruit')
    else:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[plt.cm.tab10(lbl % 10)],
                       s=15, alpha=0.6, label=f'C{lbl+1}')
axes[1].set_xlabel("PC1", fontsize=12); axes[1].set_ylabel("PC2", fontsize=12)
axes[1].set_title(f"OPTICS — {n_cl_opt} clusters (ms={BEST_MS_OPT})", fontsize=13)
if n_cl_opt <= 10:
    axes[1].legend(fontsize=9, markerscale=2)
axes[1].grid(alpha=0.2)

# c. Profil biologique
if n_cl_opt >= 2:
    df_opt_bio = df_hellinger.copy()
    df_opt_bio['cluster'] = labels_optics
    df_opt_bio = df_opt_bio[df_opt_bio['cluster'] != -1]
    bio_opt = df_opt_bio.groupby('cluster')[COLS_CONC].mean()
    im3 = axes[2].imshow(bio_opt.values.T, aspect='auto', cmap='YlOrRd')
    axes[2].set_xticks(range(len(bio_opt)))
    axes[2].set_xticklabels([f'C{i+1}' for i in bio_opt.index], fontsize=10)
    axes[2].set_yticks(range(len(COLS_CONC)))
    axes[2].set_yticklabels(COLS_CONC, fontsize=8)
    plt.colorbar(im3, ax=axes[2], label='Hellinger (moy.)')
    axes[2].set_title("Profil biologique OPTICS", fontsize=13)

plt.tight_layout()
plt.savefig('figures/fig_22_optics.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nMétriques OPTICS :")
print(f"  Silhouette     : {metrics_all['OPTICS']['silhouette']:.4f}")
print(f"  Davies-Bouldin : {metrics_all['OPTICS']['davies_bouldin']:.4f}")
print(f"  Calinski-Harab.: {metrics_all['OPTICS']['calinski_harabasz']:.1f}")
print(f"  ARI vs K-means : {metrics_all['OPTICS']['ari']:.4f}")

## 4.4 Spectral Clustering (Shi & Malik, 2000)

Le Spectral Clustering utilise la **théorie des graphes spectraux** pour détecter des structures non-convexes. Il construit un graphe de similarité entre points (k-NN ou RBF), calcule le Laplacien normalisé, puis applique K-means dans l'espace des vecteurs propres.

**Paramètres clés :**
- `n_clusters` : nombre de clusters (optimisé par silhouette)
- `affinity` : type de graphe (`nearest_neighbors` utilisé ici pour la scalabilité)

**Avantages :** capture les structures non-convexes, robuste aux géométries complexes  
**Limites :** O(n²) en mémoire, nécessite k, peu scalable → sous-échantillonnage nécessaire

In [ ]:
# === Spectral Clustering ===

# Sous-échantillon (matrice d'affinité O(n²))
N_SPEC   = min(3000, len(X_clust))
rng_spec = np.random.default_rng(42)
idx_spec = rng_spec.choice(len(X_clust), N_SPEC, replace=False)
X_spec   = X_clust[idx_spec]

# Grid search k via silhouette
k_vals_spec  = range(2, 10)
spec_results = []
print("Spectral Clustering — optimisation k (sous-échantillon)...")
for k in k_vals_spec:
    sc_tmp = SpectralClustering(n_clusters=k, affinity='nearest_neighbors',
                                n_neighbors=10, random_state=42, n_jobs=-1)
    labs_sc = sc_tmp.fit_predict(X_spec)
    sil_sc  = silhouette_score(X_spec, labs_sc,
                               sample_size=min(1000, len(X_spec)), random_state=42)
    spec_results.append({'k': k, 'silhouette': sil_sc})
    print(f"  k={k} → silhouette={sil_sc:.4f}")

df_spec_grid = pd.DataFrame(spec_results)
best_k_spec  = int(df_spec_grid.loc[df_spec_grid['silhouette'].idxmax(), 'k'])
print(f"\nk optimal Spectral : {best_k_spec}")

# Application avec k optimal sur le sous-échantillon
sc_final       = SpectralClustering(n_clusters=best_k_spec, affinity='nearest_neighbors',
                                    n_neighbors=10, random_state=42, n_jobs=-1)
labels_spec_sub = sc_final.fit_predict(X_spec)

# Extension aux données complètes par KNN
from sklearn.neighbors import KNeighborsClassifier
knn_ext = KNeighborsClassifier(n_neighbors=5)
knn_ext.fit(X_spec, labels_spec_sub)
labels_spectral = knn_ext.predict(X_clust)
n_cl_spec       = len(np.unique(labels_spectral))

metrics_all['Spectral']     = compute_metrics(X_clust, labels_spectral, labels_final)
all_clusterings['Spectral'] = labels_spectral

# --- Visualisation ---
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# a. Silhouette vs k
axes[0].plot(df_spec_grid['k'], df_spec_grid['silhouette'], 'o-', color='steelblue', lw=2)
axes[0].axvline(best_k_spec, color='crimson', linestyle='--', label=f'k optimal={best_k_spec}')
axes[0].set_xlabel("Nombre de clusters k", fontsize=12)
axes[0].set_ylabel("Score de silhouette", fontsize=12)
axes[0].set_title("Spectral — Silhouette vs k", fontsize=13)
axes[0].legend(fontsize=11); axes[0].grid(alpha=0.3)

# b. Clusters PC1-PC2
for lbl in sorted(np.unique(labels_spectral)):
    mask = labels_spectral == lbl
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[plt.cm.Set1(lbl / max(n_cl_spec - 1, 1))],
                    s=15, alpha=0.6, label=f'C{lbl+1}')
axes[1].set_xlabel("PC1", fontsize=12); axes[1].set_ylabel("PC2", fontsize=12)
axes[1].set_title(f"Spectral Clustering — {n_cl_spec} clusters", fontsize=13)
axes[1].legend(fontsize=9, markerscale=2); axes[1].grid(alpha=0.2)

# c. Profil biologique
df_spec_bio = df_hellinger.copy()
df_spec_bio['cluster'] = labels_spectral
bio_spec = df_spec_bio.groupby('cluster')[COLS_CONC].mean()
im4 = axes[2].imshow(bio_spec.values.T, aspect='auto', cmap='YlOrRd')
axes[2].set_xticks(range(len(bio_spec)))
axes[2].set_xticklabels([f'C{i+1}' for i in bio_spec.index], fontsize=10)
axes[2].set_yticks(range(len(COLS_CONC)))
axes[2].set_yticklabels(COLS_CONC, fontsize=8)
plt.colorbar(im4, ax=axes[2], label='Hellinger (moy.)')
axes[2].set_title("Profil biologique Spectral", fontsize=13)

plt.tight_layout()
plt.savefig('figures/fig_23_spectral.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nMétriques Spectral :")
print(f"  Silhouette     : {metrics_all['Spectral']['silhouette']:.4f}")
print(f"  Davies-Bouldin : {metrics_all['Spectral']['davies_bouldin']:.4f}")
print(f"  Calinski-Harab.: {metrics_all['Spectral']['calinski_harabasz']:.1f}")
print(f"  ARI vs K-means : {metrics_all['Spectral']['ari']:.4f}")

## 4.5 Gaussian Mixture Models : GMM (Fraley & Raftery, 2002)

Les GMM modélisent les données comme un **mélange de distributions gaussiennes**. Contrairement aux méthodes dures (K-means, DBSCAN), ils produisent une **assignation probabiliste** : chaque point reçoit une probabilité d'appartenance à chaque composante.

**Sélection du modèle :**
- **BIC** (Bayesian Information Criterion) : pénalise la complexité → favorise les modèles parcimonieux
- **AIC** (Akaike Information Criterion) : moins pénalisant, favorise l'ajustement

**Structures de covariance testées :** `full`, `tied`, `diag`, `spherical`

**Avantages :** doux, probabiliste, flexibilité de la covariance, sélection automatique via BIC/AIC  
**Limites :** suppose la normalité des composantes, nécessite k, sensible à l'initialisation

In [ ]:
# === GMM — Gaussian Mixture Models ===

k_vals_gmm = range(2, 11)
cov_types   = ['full', 'tied', 'diag', 'spherical']
gmm_results = []

print("GMM — sélection via BIC/AIC (grille k × covariance)...")
for k in k_vals_gmm:
    for cov in cov_types:
        try:
            gmm = GaussianMixture(n_components=k, covariance_type=cov,
                                  random_state=42, max_iter=200, n_init=3)
            gmm.fit(X_clust)
            bic  = gmm.bic(X_clust)
            aic  = gmm.aic(X_clust)
            labs = gmm.predict(X_clust)
            n_cl = len(np.unique(labs))
            sil  = silhouette_score(X_clust, labs,
                                    sample_size=min(3000, len(X_clust)), random_state=42) if n_cl >= 2 else np.nan
            gmm_results.append({'k': k, 'covariance_type': cov,
                                 'BIC': bic, 'AIC': aic,
                                 'silhouette': sil, 'converged': gmm.converged_})
        except Exception:
            gmm_results.append({'k': k, 'covariance_type': cov,
                                 'BIC': np.nan, 'AIC': np.nan,
                                 'silhouette': np.nan, 'converged': False})

df_gmm = pd.DataFrame(gmm_results)
df_gmm_ok = df_gmm[df_gmm['converged'] & df_gmm['BIC'].notna()]

# Sélection : min BIC
best_gmm_row = df_gmm_ok.loc[df_gmm_ok['BIC'].idxmin()]
BEST_K_GMM   = int(best_gmm_row['k'])
BEST_COV_GMM = best_gmm_row['covariance_type']
print(f"Meilleur modèle GMM : k={BEST_K_GMM}, covariance='{BEST_COV_GMM}'")
print(f"  BIC={best_gmm_row['BIC']:.1f}, AIC={best_gmm_row['AIC']:.1f}")

# Application finale
gmm_final = GaussianMixture(n_components=BEST_K_GMM, covariance_type=BEST_COV_GMM,
                             random_state=42, max_iter=200, n_init=5)
gmm_final.fit(X_clust)
labels_gmm = gmm_final.predict(X_clust)
proba_gmm  = gmm_final.predict_proba(X_clust)
max_proba  = proba_gmm.max(axis=1)

metrics_all['GMM']        = compute_metrics(X_clust, labels_gmm, labels_final)
all_clusterings['GMM']    = labels_gmm

# --- Visualisation ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

colors_cov = {'full': '#e41a1c', 'tied': '#377eb8', 'diag': '#4daf4a', 'spherical': '#ff7f00'}

# a. BIC
for cov in cov_types:
    sub = df_gmm_ok[df_gmm_ok['covariance_type'] == cov].sort_values('k')
    if len(sub):
        axes[0, 0].plot(sub['k'], sub['BIC'], 'o-', label=cov, color=colors_cov[cov])
axes[0, 0].axvline(BEST_K_GMM, color='black', linestyle=':', alpha=0.7)
axes[0, 0].set_xlabel("k (n_components)", fontsize=12)
axes[0, 0].set_ylabel("BIC", fontsize=12)
axes[0, 0].set_title("GMM — Sélection via BIC", fontsize=13)
axes[0, 0].legend(fontsize=10); axes[0, 0].grid(alpha=0.3)

# b. AIC
for cov in cov_types:
    sub = df_gmm_ok[df_gmm_ok['covariance_type'] == cov].sort_values('k')
    if len(sub):
        axes[0, 1].plot(sub['k'], sub['AIC'], 'o-', label=cov, color=colors_cov[cov])
axes[0, 1].axvline(BEST_K_GMM, color='black', linestyle=':', alpha=0.7)
axes[0, 1].set_xlabel("k (n_components)", fontsize=12)
axes[0, 1].set_ylabel("AIC", fontsize=12)
axes[0, 1].set_title("GMM — Sélection via AIC", fontsize=13)
axes[0, 1].legend(fontsize=10); axes[0, 1].grid(alpha=0.3)

# c. Clusters PC1-PC2
for lbl in sorted(np.unique(labels_gmm)):
    mask = labels_gmm == lbl
    axes[1, 0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                       c=[plt.cm.Set1(lbl / max(BEST_K_GMM - 1, 1))],
                       s=15, alpha=0.6, label=f'C{lbl+1}')
axes[1, 0].set_xlabel("PC1", fontsize=12); axes[1, 0].set_ylabel("PC2", fontsize=12)
axes[1, 0].set_title(f"GMM — {BEST_K_GMM} composantes ({BEST_COV_GMM})", fontsize=13)
axes[1, 0].legend(fontsize=9, markerscale=2); axes[1, 0].grid(alpha=0.2)

# d. Incertitude d'assignation
sc = axes[1, 1].scatter(X_pca[:, 0], X_pca[:, 1], c=max_proba,
                         cmap='RdYlGn', s=10, alpha=0.6, vmin=0.5, vmax=1.0)
plt.colorbar(sc, ax=axes[1, 1], label='P(meilleur cluster)')
axes[1, 1].set_xlabel("PC1", fontsize=12); axes[1, 1].set_ylabel("PC2", fontsize=12)
axes[1, 1].set_title("GMM — Incertitude d'assignation", fontsize=13)
axes[1, 1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('figures/fig_24_gmm.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nMétriques GMM :")
print(f"  Silhouette     : {metrics_all['GMM']['silhouette']:.4f}")
print(f"  Davies-Bouldin : {metrics_all['GMM']['davies_bouldin']:.4f}")
print(f"  Calinski-Harab.: {metrics_all['GMM']['calinski_harabasz']:.1f}")
print(f"  ARI vs K-means : {metrics_all['GMM']['ari']:.4f}")
print(f"  Incertitude moy. (1-P): {(1 - max_proba).mean():.4f}")

## 4.6 Comparaison Globale des Clusterings

Cette section compare systématiquement tous les algorithmes selon trois axes :
- **Métriques internes** : Silhouette (↑), Davies-Bouldin (↓), Calinski-Harabasz (↑)
- **Métriques externes** vs K-means baseline : ARI, NMI, V-measure (valeurs entre 0 et 1)
- **Structure visuelle** : projection dans le plan PC1–PC2

In [ ]:
# === Tableau comparatif global des métriques ===

methods_order_cmp = ['K-means', 'DBSCAN', 'HDBSCAN', 'OPTICS', 'Spectral', 'GMM']
rows_cmp = []
for method in methods_order_cmp:
    if method not in metrics_all:
        continue
    m = metrics_all[method]
    noise_n = m['n_noise'] if method != 'K-means' else 0
    noise_pct = f"{noise_n/len(X_clust)*100:.1f}%" if noise_n > 0 else "0%"
    rows_cmp.append({
        'Méthode'             : method,
        'k'                   : m['n_clusters'],
        'Bruit (%)'           : noise_pct,
        'Silhouette ↑'        : f"{m['silhouette']:.4f}"        if not np.isnan(m['silhouette'])        else '—',
        'Davies-Bouldin ↓'    : f"{m['davies_bouldin']:.4f}"    if not np.isnan(m['davies_bouldin'])    else '—',
        'Calinski-Harabasz ↑' : f"{m['calinski_harabasz']:.1f}" if not np.isnan(m['calinski_harabasz']) else '—',
        'ARI vs baseline'     : f"{m['ari']:.4f}"               if not np.isnan(m['ari'])               else '—',
        'NMI vs baseline'     : f"{m['nmi']:.4f}"               if not np.isnan(m['nmi'])               else '—',
        'V-measure'           : f"{m['v_measure']:.4f}"         if not np.isnan(m['v_measure'])         else '—',
    })

df_compare = pd.DataFrame(rows_cmp)
print("=" * 90)
print("TABLEAU COMPARATIF — Métriques de clustering (Jalon 3)")
print("=" * 90)
print(df_compare.to_string(index=False))
print("\nLégende :")
print("  ↑ = plus grand est meilleur | ↓ = plus petit est meilleur")
print("  ARI/NMI/V-measure : accord avec le K-means baseline (1=identique, 0=indépendant)")
print("  Bruit : points non assignés (propre aux méthodes de densité)")

In [ ]:
# === Visualisation comparative globale ===

fig, axes = plt.subplots(2, 3, figsize=(21, 12))

methods_order_plot = ['K-means', 'DBSCAN', 'HDBSCAN', 'OPTICS', 'Spectral', 'GMM']
methods_avail = [m for m in methods_order_plot if m in metrics_all]

# --- a. Heatmap métriques internes (normalisées) ---
sil_vals  = [metrics_all[m]['silhouette']         for m in methods_avail]
db_vals   = [metrics_all[m]['davies_bouldin']      for m in methods_avail]
ch_vals   = [metrics_all[m]['calinski_harabasz']   for m in methods_avail]
db_inv    = [1/v if (not np.isnan(v) and v > 0) else np.nan for v in db_vals]
metrics_matrix = np.array([sil_vals, db_inv, ch_vals]).T

metrics_norm = np.zeros_like(metrics_matrix)
for j in range(metrics_matrix.shape[1]):
    col = metrics_matrix[:, j]
    valid = ~np.isnan(col)
    if valid.sum() > 0:
        cmin, cmax = col[valid].min(), col[valid].max()
        if cmax > cmin:
            metrics_norm[:, j] = np.where(valid, (col - cmin) / (cmax - cmin), 0)
        else:
            metrics_norm[:, j] = np.where(valid, 0.5, 0)

im = axes[0, 0].imshow(metrics_norm, aspect='auto', cmap='YlGn', vmin=0, vmax=1)
axes[0, 0].set_xticks([0, 1, 2])
axes[0, 0].set_xticklabels(['Silhouette ↑', '1/DB ↑', 'CH ↑'], fontsize=10)
axes[0, 0].set_yticks(range(len(methods_avail)))
axes[0, 0].set_yticklabels(methods_avail, fontsize=11)
for (i, j), nval in np.ndenumerate(metrics_norm):
    raw = metrics_matrix[i, j]
    txt = f'{raw:.3f}' if not np.isnan(raw) else '—'
    axes[0, 0].text(j, i, txt, ha='center', va='center', fontsize=9,
                    color='black' if nval < 0.7 else 'white')
axes[0, 0].set_title("Métriques internes (normalisées [0–1])", fontsize=12)

# --- b. Heatmap métriques externes vs K-means ---
ext_matrix = np.array([
    [metrics_all[m]['ari'], metrics_all[m]['nmi'], metrics_all[m]['v_measure']]
    for m in methods_avail
])
im2 = axes[0, 1].imshow(np.nan_to_num(ext_matrix, nan=0), aspect='auto', cmap='Blues', vmin=0, vmax=1)
axes[0, 1].set_xticks([0, 1, 2])
axes[0, 1].set_xticklabels(['ARI', 'NMI', 'V-measure'], fontsize=10)
axes[0, 1].set_yticks(range(len(methods_avail)))
axes[0, 1].set_yticklabels(methods_avail, fontsize=11)
for (i, j), val in np.ndenumerate(ext_matrix):
    txt = f'{val:.3f}' if not np.isnan(val) else '—'
    axes[0, 1].text(j, i, txt, ha='center', va='center', fontsize=9,
                    color='black' if (np.isnan(val) or val < 0.6) else 'white')
axes[0, 1].set_title("Métriques externes vs K-means baseline", fontsize=12)

# --- c. Nombre de clusters ---
k_vals_bar = [metrics_all[m]['n_clusters'] for m in methods_avail]
colors_bar = ['#4393c3', '#d6604d', '#74add1', '#f4a582', '#fdae61', '#a6d96a']
bars = axes[0, 2].barh(methods_avail, k_vals_bar, color=colors_bar[:len(methods_avail)])
axes[0, 2].set_xlabel("Nombre de clusters", fontsize=11)
axes[0, 2].set_title("Nombre de clusters par méthode", fontsize=12)
for i, v in enumerate(k_vals_bar):
    axes[0, 2].text(v + 0.1, i, str(v), va='center', fontsize=10)
axes[0, 2].grid(axis='x', alpha=0.3)

# --- d/e/f. Clusters PC1-PC2 pour K-means, DBSCAN et GMM ---
subplot_methods = [m for m in ['K-means', 'DBSCAN', 'GMM'] if m in all_clusterings]
subplot_positions = [(1, 0), (1, 1), (1, 2)]
for (row, col), method in zip(subplot_positions, subplot_methods):
    labs = all_clusterings[method]
    unique_l = sorted(set(labs))
    for lbl in unique_l:
        mask = labs == lbl
        if lbl == -1:
            axes[row, col].scatter(X_pca[mask, 0], X_pca[mask, 1],
                                  c='lightgray', s=5, alpha=0.2, label='Bruit')
        else:
            axes[row, col].scatter(X_pca[mask, 0], X_pca[mask, 1],
                                  c=[plt.cm.tab10(lbl % 10)], s=10, alpha=0.5,
                                  label=f'C{lbl+1}')
    axes[row, col].set_xlabel("PC1", fontsize=10)
    axes[row, col].set_ylabel("PC2", fontsize=10)
    n_cl = metrics_all[method]['n_clusters']
    axes[row, col].set_title(f"{method} — {n_cl} clusters", fontsize=12)
    if n_cl <= 8:
        axes[row, col].legend(fontsize=8, markerscale=2)
    axes[row, col].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('figures/fig_25_comparaison_globale.png', dpi=120, bbox_inches='tight')
plt.show()

<a id="conclusions"></a>

---

## 8. Conclusions Générales

### Synthèse des trois jalons

Ce projet a conduit une analyse complète des concentrations de planctons marins à partir
des données SEANOE (Panaïotis et al., 2023), couvrant 37 campagnes océanographiques (2008–2019).

**Jalon 1 : Exploration & Baseline**  
La transformation de Hellinger et l'ACP ont permis de réduire la dimensionnalité des données
de concentration. La classification K-means baseline a identifié des groupes distincts
correspondant principalement aux couches océaniques (épipelagique vs. mésopelagique).

**Jalon 2 : Représentations Alternatives**  
L'AFC, Isomap, LLE et MDS ont révélé des structures non-linéaires complémentaires.
L'Isomap et le LLE ont notamment mis en évidence des gradients continus difficiles
à capturer par des méthodes linéaires.

**Jalon 3 : Clusterings Avancés**  
DBSCAN, HDBSCAN et OPTICS ont identifié des clusters de densité variable et isolé les
observations atypiques (bruit). Le Spectral Clustering et les GMM ont affiné la segmentation
en exploitant respectivement la connectivité spectrale et les distributions gaussiennes.

### Tableau de bord final
| Méthode | Type | Avantage |
|---|---|---|
| K-means | Partitionnel | Rapide, interprétable |
| DBSCAN | Densité | Détecte le bruit, formes arbitraires |
| HDBSCAN | Densité hiérarchique | Robuste, pas de k à fixer |
| OPTICS | Densité variable | Détection multi-échelle |
| Spectral | Graphe | Clusters non-convexes |
| GMM | Probabiliste | Affectation souple |

### Références
- Panaïotis, T. et al. (2023). Content-aware segmentation of objects spanning a large size range. *Frontiers in Marine Science*, 10. https://doi.org/10.3389/fmars.2023.1280510
- Legendre, P. & Gallagher, E.D. (2001). Ecologically meaningful transformations for ordination of species data. *Oecologia*, 129(2), 271–280.
- Tenenbaum, J.B., de Silva, V. & Langford, J.C. (2000). A global geometric framework for nonlinear dimensionality reduction. *Science*, 290(5500), 2319–2323.
- Roweis, S.T. & Saul, L.K. (2000). Nonlinear dimensionality reduction by locally linear embedding. *Science*, 290(5500), 2323–2326.
- Ester, M. et al. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. *KDD*, 96(34), 226–231.
- Campello, R.J.G.B. et al. (2013). Density-based clustering based on hierarchical density estimates. *PAKDD*, 160–172.
- Fraley, C. & Raftery, A.E. (2002). Model-based clustering, discriminant analysis, and density estimation. *JASA*, 97(458), 611–631.
